# 04 — LOVM Benchmark Experiment with VLM Router

This notebook is a **template** for running your VLM router on the **LOVM benchmark datasets**.

It will help you:

1. Use LOVM metadata (classnames + templates) to set up **zero-shot classification**.
2. Run **your VLMs** on selected LOVM datasets and log their performance in a Cauldron-style wide format.
3. Run **your router** on that logged data.
4. Compare against an **oracle LOVM-style baseline** (best static model per dataset).
5. Visualize **accuracy vs cost** per dataset using Plotly.

⚠️ You still need to fill in three implementation hooks:

- How to **load LOVM datasets** (images + labels) in your environment.
- How to **run each VLM** in zero-shot classification and compute cost.
- How to **call your trained router** to pick a model per sample.

All those parts are clearly marked with `TODO` comments.


In [1]:
import os
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import pandas as pd

import plotly.graph_objects as go

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 200)

## 1. Paths and Model Configuration

In [11]:
# ---- PATHS ----
PROJECT_ROOT = Path.cwd().parent.parent.resolve()
PROJECT_ROOT

PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router')

In [12]:
# Adjust this to where you cloned the LOVM repo:
# e.g., PROJECT_ROOT/external/LOVM
LOVM_ROOT = PROJECT_ROOT  / "code_base" / "lovm" / "LOVM"
LOVM_ROOT

DATA_OUT_DIR = PROJECT_ROOT / "dataset" / "lovm_router_eval"
DATA_OUT_DIR.mkdir(parents=True, exist_ok=True)


In [13]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOVM_ROOT:", LOVM_ROOT)
print("DATA_OUT_DIR:", DATA_OUT_DIR)

PROJECT_ROOT: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router
LOVM_ROOT: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/code_base/lovm/LOVM
DATA_OUT_DIR: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/lovm_router_eval


In [14]:

# ---- DATASETS TO USE FROM LOVM ----
# You can start with a small subset and expand.
TARGET_DATASETS: List[str] = [
    "imagenet",
    "eurosat",
    "food101",
    # add/remove based on what you want to run
]

In [15]:
# ---- YOUR MODELS ----
# Names should match the prefixes in your Cauldron-style logs, e.g.:
# 'deepseek_ocr__is_correct', 'qwen2_5_vl_3b__cost', etc.
MODEL_CONFIGS: List[Dict[str, Any]] = [
    {
        "name": "deepseek_ocr",
        "display_name": "DeepSeek OCR VLM",
    },
    {
        "name": "qwen2_5_vl_3b",
        "display_name": "Qwen2.5 VL 3B",
    },
    {
        "name": "qwen2_5_vl_7b",
        "display_name": "Qwen2.5 VL 7B",
    },
    {
        "name": "qwen3_vl_8b_thinking",
        "display_name": "Qwen3 VL 8B (thinking)",
    },
    {
        "name": "gemma_3_27b",
        "display_name": "Gemma 3 27B VLM",
    },
]

MODEL_NAMES = [m["name"] for m in MODEL_CONFIGS]
MODEL_NAMES

['deepseek_ocr',
 'qwen2_5_vl_3b',
 'qwen2_5_vl_7b',
 'qwen3_vl_8b_thinking',
 'gemma_3_27b']

## 2. LOVM Metadata: Classnames and Templates

In [17]:
def load_lovm_classnames(ds_name: str) -> List[str]:
    """Load class names for a LOVM dataset from classnames/*.txt."""
    path = LOVM_ROOT / "classnames" / f"{ds_name}.txt"
    if not path.exists():
        raise FileNotFoundError(f"Classnames file not found for {ds_name}: {path}")
    with open(path, "r", encoding="utf-8") as f:
        classnames = [line.strip() for line in f if line.strip()]
    return classnames


def load_lovm_templates(ds_name: str) -> List[str]:
    """Load templates for a LOVM dataset from templates/*.txt."""
    path = LOVM_ROOT / "templates" / f"{ds_name}.txt"
    if not path.exists():
        raise FileNotFoundError(f"Templates file not found for {ds_name}: {path}")
    with open(path, "r", encoding="utf-8") as f:
        templates = [line.strip() for line in f if line.strip()]
    return templates


def build_zero_shot_prompts(classnames: List[str], templates: List[str]) -> List[str]:
    """
    Build zero-shot prompts like CLIP:
    template: "a photo of a {}" or "{label}" etc.
    """
    prompts = []
    for cname in classnames:
        for t in templates:
            if "{}" in t:
                prompts.append(t.format(cname))
            elif "{label}" in t:
                prompts.append(t.replace("{label}", cname))
            else:
                # fallback: append label at end
                prompts.append(f"{t} {cname}")
    return prompts

## 3. Dataset Loader (TODO)

You need to implement a loader that, for a given LOVM dataset name, yields:

```python
(sample_id, image, label_idx)
```

- `sample_id`: any unique string (e.g., `"imagenet_000123"`)
- `image`: a PIL image, tensor, or whatever your model expects
- `label_idx`: integer index into the `classnames` list


In [18]:
def load_lovm_dataset_images(ds_name: str):
    """
    Return an iterable of (sample_id, image, label_idx) for the given LOVM dataset.

    TODO: IMPLEMENT this for your environment.

    Options:
      - Use CLIP-benchmark dataset builder
      - Use torchvision / custom loaders mapped to LOVM names
    """
    # Example sketch (pseudocode):
    #
    # from clip_benchmark.datasets.builder import build_dataset
    # dataset = build_dataset(
    #     dataset=ds_name,
    #     root="/path/to/lovm_datasets",
    #     split="test",  # or "val"
    # )
    # for i, (image, label_idx) in enumerate(dataset):
    #     sample_id = f"{ds_name}_{i:06d}"
    #     yield sample_id, image, label_idx
    #
    raise NotImplementedError("Implement load_lovm_dataset_images(ds_name)")

## 4. VLM Inference (TODO)

We define a helper that runs **one model** on **one sample** in zero-shot classification mode,
using LOVM classnames + templates.

You should plug in your own inference + cost utilities here.


In [ ]:
def run_single_model_zero_shot(
    model_name: str,
    image,
    classnames: List[str],
    templates: List[str],
    label_idx: int,
) -> Dict[str, Any]:
    """
    Run a single VLM in zero-shot classification mode.

    Returns a dict with at least:
      {
        "pred_idx": int,
        "is_correct": int (0/1),
        "cost": float,
        "valid_mask": bool,
        "score_f1": float (optional),
        "sample_score": float (optional)
      }

    TODO: implement using your inference + cost_utils + performance_utils.
    """
    # Example sketch:
    #
    # prompts = build_zero_shot_prompts(classnames, templates)
    # scores = my_vlm_inference(model_name, image, prompts)
    # pred_idx = argmax over classes
    # is_correct = int(pred_idx == label_idx)
    # cost = compute_cost(tokens_in, tokens_out, model_name)
    #
    # return {
    #     "pred_idx": int(pred_idx),
    #     "is_correct": is_correct,
    #     "cost": float(cost),
    #     "valid_mask": True,
    #     "score_f1": float(is_correct),  # or something more advanced
    #     "sample_score": float(is_correct),
    # }
    raise NotImplementedError("Implement run_single_model_zero_shot(model_name, image, ...)"

## 5. Collect Evaluation Rows for All Models and Datasets

This will:

- Loop over `TARGET_DATASETS`.
- For each `(sample_id, image, label_idx)`,
- Run each model in `MODEL_CONFIGS`,
- Log results in a **wide** dataframe (one row per sample, columns per model).

In [ ]:
def collect_lovm_eval_rows() -> pd.DataFrame:
    """Run all models on all selected LOVM datasets and build a wide-format dataframe."""
    rows = []

    for ds_name in TARGET_DATASETS:
        print(f"\n=== Dataset: {ds_name} ===")
        classnames = load_lovm_classnames(ds_name)
        templates = load_lovm_templates(ds_name)

        for sample_id, image, label_idx in load_lovm_dataset_images(ds_name):
            base_row = {
                "sample_id": sample_id,
                "image_path": None,  # or path if you save images to disk
                "prompt_raw": f"zero-shot classification on {ds_name}",
                "router_task": ds_name,
                "source_dataset": "lovm",
                "source_config": ds_name,
                "ground_truth": classnames[label_idx],
                "ground_truth_type": "mc",
            }

            for m in MODEL_CONFIGS:
                name = m["name"]
                try:
                    out = run_single_model_zero_shot(
                        model_name=name,
                        image=image,
                        classnames=classnames,
                        templates=templates,
                        label_idx=label_idx,
                    )
                    base_row[f"{name}__is_correct"]   = int(out.get("is_correct", 0))
                    base_row[f"{name}__cost"]         = float(out.get("cost", 0.0))
                    base_row[f"{name}__valid_mask"]   = bool(out.get("valid_mask", True))
                    base_row[f"{name}__score_f1"]     = float(out.get("score_f1", out.get("is_correct", 0)))
                    base_row[f"{name}__sample_score"] = float(out.get("sample_score", out.get("is_correct", 0)))
                except Exception as e:
                    # If model fails on this sample, mark invalid
                    base_row[f"{name}__is_correct"]   = 0
                    base_row[f"{name}__cost"]         = 0.0
                    base_row[f"{name}__valid_mask"]   = False
                    base_row[f"{name}__score_f1"]     = 0.0
                    base_row[f"{name}__sample_score"] = 0.0

            rows.append(base_row)

    df = pd.DataFrame(rows)
    return df


# Uncomment these lines when you're ready to run the (potentially long) collection:

# df_lovm_eval = collect_lovm_eval_rows()
# df_lovm_eval.to_parquet(DATA_OUT_DIR / "lovm_eval_all_models.parquet", index=False)
# df_lovm_eval.head()

## 6. Run Your Router on the LOVM Eval Data (TODO)

Here, you plug in your trained router to choose `router_best_model_name` for each sample,
then derive:

- `router_is_correct` (from chosen model's `__is_correct`)
- `router_chosen_cost` (from chosen model's `__cost`)
- `router_chosen_perf` (e.g., from `__sample_score` or a custom utility).


In [ ]:
def run_router_on_df(df: pd.DataFrame) -> pd.DataFrame:
    """Use your router to decide a model per sample and log router metrics."""
    # TODO: replace this placeholder with a call to your trained router.
    # For now, this just picks a fixed model as a dummy baseline.
    router_best_names = []
    router_is_correct = []
    router_cost = []
    router_perf = []

    for idx, row in df.iterrows():
        # TODO: call your router model here (e.g., my_router_predict(row))
        # Example placeholder:
        chosen = "gemma_3_27b"

        router_best_names.append(chosen)

        corr_col = f"{chosen}__is_correct"
        cost_col = f"{chosen}__cost"
        score_col = f"{chosen}__sample_score"

        corr_val = float(row.get(corr_col, 0.0) or 0.0)
        cost_val = float(row.get(cost_col, 0.0) or 0.0)
        perf_val = float(row.get(score_col, corr_val))

        router_is_correct.append(corr_val)
        router_cost.append(cost_val)
        router_perf.append(perf_val)

    df["router_best_model_name"] = router_best_names
    df["router_is_correct"] = router_is_correct
    df["router_chosen_cost"] = router_cost
    df["router_chosen_perf"] = router_perf

    return df


# Example usage (after you have lovm_eval_all_models.parquet):
# df_lovm_eval = pd.read_parquet(DATA_OUT_DIR / "lovm_eval_all_models.parquet")
# df_lovm_router = run_router_on_df(df_lovm_eval)
# df_lovm_router.to_parquet(DATA_OUT_DIR / "lovm_eval_with_router.parquet", index=False)
# df_lovm_router.head()

## 7. Oracle LOVM-Style Baseline: Best Static Model per Task

This is an **oracle LOVM-style baseline**:

- For each dataset (`router_task`),
- Find the model with the **highest accuracy**,
- Pretend we always use that model for that dataset.


In [ ]:
def summarize_model_on_subset(sub_df: pd.DataFrame, model_name: str) -> Dict[str, Any]:
    col_corr  = f"{model_name}__is_correct"
    col_cost  = f"{model_name}__cost"
    col_valid = f"{model_name}__valid_mask"

    if col_corr not in sub_df.columns:
        raise ValueError(f"Missing {col_corr}")
    if col_cost not in sub_df.columns:
        raise ValueError(f"Missing {col_cost}")

    if col_valid in sub_df.columns:
        valid_mask = sub_df[col_valid].fillna(False).astype(bool)
    else:
        valid_mask = pd.Series(True, index=sub_df.index)

    valid_df = sub_df[valid_mask]
    num_samples = len(sub_df)
    num_valid = len(valid_df)

    if num_valid == 0:
        return {
            "model_name": model_name,
            "num_samples": num_samples,
            "num_valid": 0,
            "accuracy": np.nan,
            "avg_cost_per_sample": np.nan,
            "total_cost": np.nan,
        }

    corr = valid_df[col_corr].fillna(0.0).astype(float)
    cost = valid_df[col_cost].fillna(0.0).astype(float)

    return {
        "model_name": model_name,
        "num_samples": num_samples,
        "num_valid": num_valid,
        "accuracy": float(corr.mean()),
        "avg_cost_per_sample": float(cost.mean()),
        "total_cost": float(cost.sum()),
    }


def build_model_task_perf(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for task, task_df in df.groupby("router_task"):
        for name in MODEL_NAMES:
            s = summarize_model_on_subset(task_df, name)
            s["task"] = task
            rows.append(s)
    out = pd.DataFrame(rows)
    out = out[[
        "task", "model_name", "num_samples", "num_valid",
        "accuracy", "avg_cost_per_sample", "total_cost",
    ]]
    return out

In [ ]:
# Example usage once you have df_lovm_router:

# df_lovm_router = pd.read_parquet(DATA_OUT_DIR / "lovm_eval_with_router.parquet")
# model_task_perf = build_model_task_perf(df_lovm_router)
# model_task_perf.head()

## 8. Compute Oracle per Sample and Summaries (Overall + Per Task)

In [ ]:
def compute_oracle_and_summaries(df_lovm_router: pd.DataFrame):
    # Build per-task, per-model performance
    model_task_perf = build_model_task_perf(df_lovm_router)

    # For each task, pick model with highest accuracy
    idx_best = model_task_perf.groupby("task")["accuracy"].idxmax()
    lovm_oracle_baseline = model_task_perf.loc[idx_best].copy().reset_index(drop=True)

    # Map task -> chosen model
    task_to_oracle_model = dict(zip(lovm_oracle_baseline["task"], lovm_oracle_baseline["model_name"]))

    oracle_is_correct = []
    oracle_cost = []
    oracle_model_for_sample = []

    for _, row in df_lovm_router.iterrows():
        task = row["router_task"]
        chosen = task_to_oracle_model[task]
        col_corr = f"{chosen}__is_correct"
        col_cost = f"{chosen}__cost"

        corr_val = float(row.get(col_corr, 0.0) or 0.0)
        cost_val = float(row.get(col_cost, 0.0) or 0.0)

        oracle_is_correct.append(corr_val)
        oracle_cost.append(cost_val)
        oracle_model_for_sample.append(chosen)

    df_lovm_router["oracle_model_name"] = oracle_model_for_sample
    df_lovm_router["oracle_is_correct"] = oracle_is_correct
    df_lovm_router["oracle_cost"] = oracle_cost

    # Overall summary
    overall = pd.DataFrame({
        "method": ["oracle_static", "router"],
        "accuracy": [
            df_lovm_router["oracle_is_correct"].mean(),
            df_lovm_router["router_is_correct"].mean(),
        ],
        "avg_cost_per_sample": [
            df_lovm_router["oracle_cost"].mean(),
            df_lovm_router["router_chosen_cost"].mean(),
        ],
        "total_cost": [
            df_lovm_router["oracle_cost"].sum(),
            df_lovm_router["router_chosen_cost"].sum(),
        ],
    })

    # Per-task summary
    router_task_summary = (
        df_lovm_router
        .groupby("router_task")
        .agg(
            num_samples_router=("router_is_correct", "size"),
            accuracy_router=("router_is_correct", "mean"),
            avg_cost_per_sample_router=("router_chosen_cost", "mean"),
            total_cost_router=("router_chosen_cost", "sum"),
        )
        .reset_index()
        .rename(columns={"router_task": "task"})
    )

    per_task = lovm_oracle_baseline.merge(router_task_summary, on="task", how="inner")
    per_task = per_task[[
        "task",
        "model_name", "num_samples", "num_valid",
        "accuracy", "avg_cost_per_sample", "total_cost",
        "num_samples_router", "accuracy_router",
        "avg_cost_per_sample_router", "total_cost_router",
    ]]

    return df_lovm_router, model_task_perf, lovm_oracle_baseline, overall, per_task


# Example usage:
# df_lovm_router = pd.read_parquet(DATA_OUT_DIR / "lovm_eval_with_router.parquet")
# df_lovm_router, model_task_perf, lovm_oracle_baseline, overall, per_task = compute_oracle_and_summaries(df_lovm_router)
# display(overall)
# display(per_task.sort_values("task"))

## 9. Plot: Accuracy vs Cost per Task (Oracle Static vs Router)

This uses a log-scale x-axis for cost and shows a dotted line connecting
oracle baseline and router for each LOVM dataset.


In [ ]:
def plot_accuracy_vs_cost(per_task: pd.DataFrame):
    rows = []
    for _, r in per_task.iterrows():
        rows.append({
            "task": r["task"],
            "method": "Oracle static",
            "accuracy": r["accuracy"],
            "avg_cost_per_sample": r["avg_cost_per_sample"],
            "total_cost": r["total_cost"],
            "num_samples": r["num_samples"],
        })
        rows.append({
            "task": r["task"],
            "method": "Router",
            "accuracy": r["accuracy_router"],
            "avg_cost_per_sample": r["avg_cost_per_sample_router"],
            "total_cost": r["total_cost_router"],
            "num_samples": r["num_samples_router"],
        })

    df_plot = pd.DataFrame(rows)

    fig = go.Figure()

    # Connect oracle ↔ router points per task
    for task, g in df_plot.groupby("task"):
        if len(g) != 2:
            continue
        fig.add_trace(
            go.Scatter(
                x=g["avg_cost_per_sample"],
                y=g["accuracy"],
                mode="lines",
                line=dict(width=1, dash="dot"),
                showlegend=False,
                hoverinfo="skip",
            )
        )

    # Add the two method point sets
    for method, marker_symbol in [("Oracle static", "circle"), ("Router", "x")]:
        g = df_plot[df_plot["method"] == method].copy()
        fig.add_trace(
            go.Scatter(
                x=g["avg_cost_per_sample"],
                y=g["accuracy"],
                mode="markers+text",
                name=method,
                text=g["task"],
                textposition="top center",
                textfont=dict(size=9),
                marker=dict(size=9, line=dict(width=1)),
                hovertemplate=(
                    "<b>%{text}</b><br>"
                    "Method: %{customdata[0]}<br>"
                    "Accuracy: %{y:.3f}<br>"
                    "Avg cost/sample: %{x:.6f}<br>"
                    "Total cost: %{customdata[1]:.3f}<br>"
                    "Num samples: %{customdata[2]}<extra></extra>"
                ),
                customdata=np.stack(
                    [g["method"], g["total_cost"], g["num_samples"]],
                    axis=1
                ),
            )
        )

    fig.update_layout(
        title="LOVM Datasets: Accuracy vs Cost (Oracle Static vs Router)",
        xaxis_title="Avg cost per sample ($, log scale)",
        yaxis_title="Accuracy",
        template="plotly_white",
        legend_title_text="Method",
        xaxis=dict(
            type="log",           # log scale for readability
            tickformat=".2e",    # scientific notation (e.g. 1.00e-04)
            tickfont=dict(size=10),
            showgrid=True,
        ),
        yaxis=dict(range=[0, 1.05]),
        hovermode="closest",
    )

    fig.show()


# Example:
# plot_accuracy_vs_cost(per_task)

## 10. Next Steps

1. **Implement the TODOs**:
   - `load_lovm_dataset_images(ds_name)`
   - `run_single_model_zero_shot(...)`
   - `run_router_on_df(df)`

2. **Run the pipeline** in order:
   - Collect LOVM eval data: `collect_lovm_eval_rows()` → `lovm_eval_all_models.parquet`
   - Run router: `run_router_on_df(...)` → `lovm_eval_with_router.parquet`
   - Compute oracle + summaries: `compute_oracle_and_summaries(...)`
   - Plot with `plot_accuracy_vs_cost(per_task)`.

3. **Use the outputs** in your paper:
   - Overall table (`overall`)
   - Per-task table (`per_task`)
   - Plotly figure (accuracy vs cost).

This gives you a **clean, LOVM-aligned evaluation** of your VLM router:
- Oracle LOVM-style static baseline vs
- Your per-sample, cost-aware router.
